# Medical Symptom Triage Conversational (Final V3)

## Project Pipeline (Final)

1. **Load data**: `train` and `validation` from HF parquet/datasets.
2. **Clean text**: lowercase, normalize spaces, remove special chars, keep patient-facing text only.
3. **Merge rare specialty classes**: classes with `<50` samples in train are merged into `Other`.
4. **Feature engineering**:
   - `specialty`: MiniLM embeddings (CPU-friendly contextual representation)
   - `urgency`: TF-IDF n-grams
5. **Imbalance handling**: Random oversampling (or fallback oversampling).
6. **Modeling**:
   - `specialty`: LogReg/SVM on MiniLM embeddings
   - `urgency`: LogReg on TF-IDF
7. **Evaluation**: Accuracy, F1-weighted, MCC, Cohen Kappa, plus optional joint triage effectiveness.
8. **Inference app**: Streamlit predicts **urgency as primary output** and returns **specialty as top suggestions**.

### Why specialty is shown as suggestion (not hard decision)
The dataset is conversational and partially synthetic with overlapping symptom patterns across specialties.
Urgency is more stable for direct decisioning, while specialty prediction is best treated as ranked recommendations (top suggestions) to support, not replace, clinical routing.



## V3 Final CPU Pipeline – Specialty + Urgency

This is a final, CPU-friendly pipeline:
- `specialty` predicted from MiniLM embeddings,
- `urgency` predicted from TF-IDF,
- separate models and metrics,
- optional export to `joblib`.

In [ ]:
# V3-1) Imports
import ast
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, cohen_kappa_score

try:
    from imblearn.over_sampling import RandomOverSampler
    HAS_IMBLEARN_V3 = True
except Exception:
    HAS_IMBLEARN_V3 = False

try:
    from sentence_transformers import SentenceTransformer
    HAS_ST_V3 = True
except Exception:
    HAS_ST_V3 = False
    print("sentence-transformers not available. Install: %pip install sentence-transformers")

In [ ]:
# V3-2) Load parquet splits (or fallback to datasets)
splits = {
    "train": "data/train-00000-of-00001.parquet",
    "validation": "data/validation-00000-of-00001.parquet",
}

try:
    v3_train = pd.read_parquet("hf://datasets/sweatSmile/medical-symptom-triage-conversational/" + splits["train"])
    v3_val = pd.read_parquet("hf://datasets/sweatSmile/medical-symptom-triage-conversational/" + splits["validation"])
    print("Loaded V3 data via hf:// parquet")
except Exception:
    from datasets import load_dataset

    ds_v3 = load_dataset("sweatSmile/medical-symptom-triage-conversational")
    v3_train = ds_v3["train"].to_pandas()
    v3_val = ds_v3["validation"].to_pandas()
    print("Loaded V3 data via datasets.load_dataset fallback")

print("v3_train:", v3_train.shape)
print("v3_val:", v3_val.shape)
print("Columns:", v3_train.columns.tolist())

In [ ]:
# V3-3) Build `text`, clean, merge rare specialty classes (<50)

def parse_messages_blob_v3(blob):
    if isinstance(blob, list):
        parsed = blob
    elif isinstance(blob, str):
        try:
            parsed = ast.literal_eval(blob)
        except Exception:
            parsed = []
    else:
        parsed = []
    return parsed if isinstance(parsed, list) else []


def extract_user_text_v3(messages_blob):
    msgs = parse_messages_blob_v3(messages_blob)
    parts = []
    for m in msgs:
        if isinstance(m, dict) and m.get("role") == "user":
            txt = str(m.get("content", "")).strip()
            if txt:
                parts.append(txt)
    return "\n".join(parts)


def clean_text_v3(text):
    text = str(text).lower().strip()
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


for frame in (v3_train, v3_val):
    frame["text"] = frame["messages"].map(extract_user_text_v3)
    frame["clean_text"] = frame["text"].map(clean_text_v3)

# Remove empty and duplicate records
v3_train = v3_train[v3_train["clean_text"] != ""].copy()
v3_val = v3_val[v3_val["clean_text"] != ""].copy()

v3_train = v3_train.drop_duplicates(subset=["clean_text", "specialty", "urgency"]).reset_index(drop=True)
v3_val = v3_val.drop_duplicates(subset=["clean_text", "specialty", "urgency"]).reset_index(drop=True)

# Merge rare specialty classes by TRAIN distribution
MIN_SPEC_V3 = 50
spec_counts_v3 = v3_train["specialty"].value_counts()
stable_spec_v3 = set(spec_counts_v3[spec_counts_v3 >= MIN_SPEC_V3].index.tolist())


def map_spec_v3(x):
    return x if x in stable_spec_v3 else "Other"


v3_train["specialty_merged"] = v3_train["specialty"].map(map_spec_v3)
v3_val["specialty_merged"] = v3_val["specialty"].map(map_spec_v3)

print("After cleaning:", v3_train.shape, v3_val.shape)
print("Specialty merged distribution (train):")
display(v3_train["specialty_merged"].value_counts())
print("Urgency distribution (train):")
display(v3_train["urgency"].value_counts())

In [ ]:
# V3-4) Encode labels and build features
le_spec_v3 = LabelEncoder()
le_urg_v3 = LabelEncoder()

v3_train["specialty_label"] = le_spec_v3.fit_transform(v3_train["specialty_merged"])
v3_val["specialty_label"] = le_spec_v3.transform(v3_val["specialty_merged"])

v3_train["urgency_label"] = le_urg_v3.fit_transform(v3_train["urgency"])
v3_val["urgency_label"] = le_urg_v3.transform(v3_val["urgency"])

# Specialty input: MiniLM embeddings
if not HAS_ST_V3:
    raise ImportError("sentence-transformers is required for V3 specialty embeddings.")

emb_model_id_v3 = "sentence-transformers/all-MiniLM-L6-v2"
emb_model_v3 = SentenceTransformer(emb_model_id_v3)

X_train_spec_emb = emb_model_v3.encode(v3_train["clean_text"].tolist(), convert_to_numpy=True, show_progress_bar=True)
X_val_spec_emb = emb_model_v3.encode(v3_val["clean_text"].tolist(), convert_to_numpy=True, show_progress_bar=True)

# Urgency input: TF-IDF
tfidf_v3 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_urg_tfidf = tfidf_v3.fit_transform(v3_train["clean_text"])
X_val_urg_tfidf = tfidf_v3.transform(v3_val["clean_text"])

print("X_train_spec_emb:", X_train_spec_emb.shape)
print("X_train_urg_tfidf:", X_train_urg_tfidf.shape)

In [ ]:
# V3-5) Oversampling helpers (specialty + urgency)

def fallback_oversample_sparse(X, y, random_state=42):
    ys = pd.Series(y).reset_index(drop=True)
    max_count = ys.value_counts().max()
    rng = np.random.RandomState(random_state)

    from scipy.sparse import vstack

    X_parts, y_parts = [], []
    for cls in sorted(ys.unique()):
        idx = np.where(ys.values == cls)[0]
        if len(idx) < max_count:
            extra = rng.choice(idx, size=max_count - len(idx), replace=True)
            idx_bal = np.concatenate([idx, extra])
        else:
            idx_bal = idx
        X_parts.append(X[idx_bal])
        y_parts.append(np.full(len(idx_bal), cls))

    Xb = vstack(X_parts)
    yb = np.concatenate(y_parts)
    perm = rng.permutation(len(yb))
    return Xb[perm], yb[perm]


def fallback_oversample_dense(X, y, random_state=42):
    ys = pd.Series(y).reset_index(drop=True)
    max_count = ys.value_counts().max()
    rng = np.random.RandomState(random_state)

    X_parts, y_parts = [], []
    for cls in sorted(ys.unique()):
        idx = np.where(ys.values == cls)[0]
        if len(idx) < max_count:
            extra = rng.choice(idx, size=max_count - len(idx), replace=True)
            idx_bal = np.concatenate([idx, extra])
        else:
            idx_bal = idx
        X_parts.append(X[idx_bal])
        y_parts.append(np.full(len(idx_bal), cls))

    Xb = np.vstack(X_parts)
    yb = np.concatenate(y_parts)
    perm = rng.permutation(len(yb))
    return Xb[perm], yb[perm]


if HAS_IMBLEARN_V3:
    ros_v3 = RandomOverSampler(random_state=42)

    X_train_spec_bal, y_train_spec_bal = ros_v3.fit_resample(X_train_spec_emb, v3_train["specialty_label"])
    X_train_urg_bal, y_train_urg_bal = ros_v3.fit_resample(X_train_urg_tfidf, v3_train["urgency_label"])
    print("Used RandomOverSampler (imblearn)")
else:
    X_train_spec_bal, y_train_spec_bal = fallback_oversample_dense(X_train_spec_emb, v3_train["specialty_label"])
    X_train_urg_bal, y_train_urg_bal = fallback_oversample_sparse(X_train_urg_tfidf, v3_train["urgency_label"])
    print("Used fallback oversampling")

print("Balanced specialty:", X_train_spec_bal.shape, len(y_train_spec_bal))
print("Balanced urgency:", X_train_urg_bal.shape, len(y_train_urg_bal))

In [ ]:
# V3-6) Train models
# Specialty: choose LogReg or SVM on MiniLM embeddings
v3_specialty_models = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "SVM": SVC(kernel="linear", class_weight="balanced"),
}

v3_specialty_fitted = {}
for name, mdl in v3_specialty_models.items():
    mdl.fit(X_train_spec_bal, y_train_spec_bal)
    v3_specialty_fitted[name] = mdl

# Urgency: LogReg on TF-IDF
v3_urgency_model = LogisticRegression(max_iter=1000, class_weight="balanced")
v3_urgency_model.fit(X_train_urg_bal, y_train_urg_bal)

print("V3 models trained.")

In [ ]:
# V3-7) Evaluation + triage effectiveness score

def eval_metrics_v3(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


rows = []

# Specialty models (features = MiniLM embeddings)
for name, mdl in v3_specialty_fitted.items():
    pred_spec = mdl.predict(X_val_spec_emb)
    m = eval_metrics_v3(v3_val["specialty_label"], pred_spec)
    rows.append(
        {
            "target": "specialty",
            "feature_source": f"MiniLM embeddings ({emb_model_id_v3})",
            "model": name,
            **{k: round(v, 4) for k, v in m.items()},
        }
    )

# Urgency model (features = TF-IDF)
pred_urg = v3_urgency_model.predict(X_val_urg_tfidf)
m_urg = eval_metrics_v3(v3_val["urgency_label"], pred_urg)
rows.append(
    {
        "target": "urgency",
        "feature_source": "TF-IDF (1-2 grams)",
        "model": "LogReg",
        **{k: round(v, 4) for k, v in m_urg.items()},
    }
)

v3_results_df = pd.DataFrame(rows)
print("=== V3 RESULTS ===")
display(v3_results_df.sort_values(["target", "f1_weighted"], ascending=[True, False]))

# Optional triage effectiveness (joint exact match using best specialty model + urgency model)
best_spec_model_name = (
    v3_results_df[v3_results_df["target"] == "specialty"]
    .sort_values("f1_weighted", ascending=False)
    .iloc[0]["model"]
)
best_spec_model = v3_specialty_fitted[best_spec_model_name]

pred_spec_best = best_spec_model.predict(X_val_spec_emb)
pred_urg_best = v3_urgency_model.predict(X_val_urg_tfidf)

joint_exact = np.mean(
    (pred_spec_best == v3_val["specialty_label"].values)
    & (pred_urg_best == v3_val["urgency_label"].values)
)

print(f"Best specialty model (on MiniLM): {best_spec_model_name}")
print(f"Triage effectiveness (joint exact match): {joint_exact:.4f}")
print("\nNote: TinyBERT results are in V2 contextual block (V2-14), not in V3 by default.")

In [ ]:
# V3-7b) Optional TinyBERT run for specialty (CPU, slower)
# This adds TinyBERT results directly into V3 comparison table.

try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    HAS_TINYBERT_V3 = True
except Exception as e:
    HAS_TINYBERT_V3 = False
    print("TinyBERT dependencies missing:", e)


def mean_pool_v3(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)


if HAS_TINYBERT_V3:
    tinybert_id_v3 = "huawei-noah/TinyBERT_General_4L_312D"
    tokenizer_v3 = AutoTokenizer.from_pretrained(tinybert_id_v3)
    model_v3 = AutoModel.from_pretrained(tinybert_id_v3)
    model_v3.eval()

    def encode_tinybert_v3(texts, batch_size=32, max_length=256):
        all_vecs = []
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                enc = tokenizer_v3(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt",
                )
                out = model_v3(**enc)
                vec = mean_pool_v3(out.last_hidden_state, enc["attention_mask"])
                all_vecs.append(vec.cpu().numpy())
        return np.vstack(all_vecs)

    X_train_tiny_v3 = encode_tinybert_v3(v3_train["clean_text"].astype(str).tolist())
    X_val_tiny_v3 = encode_tinybert_v3(v3_val["clean_text"].astype(str).tolist())

    tiny_lr = LogisticRegression(max_iter=1000, class_weight="balanced")
    tiny_lr.fit(X_train_tiny_v3, v3_train["specialty_label"].values)
    tiny_pred = tiny_lr.predict(X_val_tiny_v3)
    tiny_m = eval_metrics_v3(v3_val["specialty_label"].values, tiny_pred)

    tiny_row = {
        "target": "specialty",
        "feature_source": f"TinyBERT embeddings ({tinybert_id_v3})",
        "model": "LogReg",
        **{k: round(v, 4) for k, v in tiny_m.items()},
    }

    # Upsert into table
    if "v3_results_df" in globals():
        v3_results_df = pd.concat([v3_results_df, pd.DataFrame([tiny_row])], ignore_index=True)
        print("Added TinyBERT row to V3 results.")
        display(v3_results_df.sort_values(["target", "f1_weighted"], ascending=[True, False]))
    else:
        print("Run V3-7 first to initialize v3_results_df.")

In [ ]:
# V3-8) Optional export (joblib)
# %pip install -q joblib

# from joblib import dump
# dump(best_spec_model, "v3_best_specialty_model.joblib")
# dump(v3_urgency_model, "v3_urgency_model.joblib")
# dump(le_spec_v3, "v3_le_specialty.joblib")
# dump(le_urg_v3, "v3_le_urgency.joblib")
# dump(emb_model_v3, "v3_embedding_model.joblib")  # optional, often large
# dump(tfidf_v3, "v3_tfidf_urgency.joblib")
# print("V3 artifacts exported.")

### V3 Reproducibility / Run Order
1. Install deps: `pandas`, `scikit-learn`, `sentence-transformers` (and optionally `imblearn`, `joblib`).
2. Run `V3-1` -> `V3-7`.
3. Optional: run `V3-8` for export.
4. Use exported artifacts for a Streamlit `predict(text)` endpoint returning `specialty + urgency`.

## Streamlit App (Urgency-first)

This app predicts:
- **primary:** `urgency`
- **secondary:** top specialty suggestions.

App file: `medical_triage_streamlit.py`

In [ ]:
# Run Streamlit app (urgency primary + specialty suggestions)
!python3 -m streamlit run "/Users/turfian/Mastering-NLP-from-Foundations-to-LLMs/medical_triage_streamlit.py" --server.address 127.0.0.1 --server.port 8503